# Toy JEPA Demo: Predicting a Missing Word in Latent Space

This notebook demonstrates the **idea** behind JEPA (Joint-Embedding Predictive Architecture) using a tiny synthetic text dataset.

The goal is not to build a production-grade JEPA model. The goal is to show the core learning pattern:

1. Observe the visible context.
2. Encode it into a latent representation.
3. Encode the hidden target into another latent representation.
4. Train a predictor to match the target latent from the context latent.

Unlike a standard classifier, we do **not** directly train the model to emit the missing word. We first train it to predict the **embedding** of the missing word. After training, we recover an interpretable word by nearest-neighbor search in latent space.

This notebook is designed to run in **Google Colab** or a local **Jupyter Notebook** with PyTorch installed.

In [ ]:
import math
import random
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## Build a Tiny Dataset

We will use short sentences with simple world structure.

Each training example removes one word and asks the model to infer the missing concept from the remaining context.

In [ ]:
sentences = [
    "the cat climbed the tree",
    "the cat jumped on the wall",
    "the dog chased the ball",
    "the bird sat on the branch",
    "the child opened the door",
    "the child closed the window",
    "the rabbit hid in the bush",
    "the monkey climbed the tree",
    "the squirrel ran up the tree",
    "the fish swam in the pond",
    "the boat crossed the river",
    "the student opened the book",
    "the teacher wrote on the board",
    "the chef cut the bread",
    "the driver turned the wheel",
]

examples = []
for sentence in sentences:
    tokens = sentence.split()
    for idx in range(1, len(tokens)):
        if tokens[idx] in {"the", "on", "in", "up"}:
            continue
        context = [tok for j, tok in enumerate(tokens) if j != idx]
        target = tokens[idx]
        examples.append((context, target, sentence, idx))

len(examples), examples[:5]

In [ ]:
vocab = sorted({tok for sentence in sentences for tok in sentence.split()})
stoi = {token: i for i, token in enumerate(vocab)}
itos = {i: token for token, i in stoi.items()}

PAD = "<pad>"
if PAD not in stoi:
    stoi = {PAD: 0, **{token: i + 1 for token, i in stoi.items()}}
    itos = {i: token for token, i in stoi.items()}

vocab_size = len(stoi)
vocab_size, list(stoi.items())[:10]

In [ ]:
max_context_len = max(len(context) for context, _, _, _ in examples)
max_context_len

## Dataset and DataLoader

In [ ]:
class MissingWordDataset(Dataset):
    def __init__(self, rows, stoi, max_context_len):
        self.rows = rows
        self.stoi = stoi
        self.max_context_len = max_context_len

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        context, target, sentence, missing_index = self.rows[idx]
        context_ids = [self.stoi[t] for t in context]
        context_ids = context_ids + [self.stoi[PAD]] * (self.max_context_len - len(context_ids))
        target_id = self.stoi[target]
        return {
            "context_ids": torch.tensor(context_ids, dtype=torch.long),
            "target_id": torch.tensor(target_id, dtype=torch.long),
            "sentence": sentence,
            "missing_index": missing_index,
            "target_word": target,
        }

dataset = MissingWordDataset(examples, stoi, max_context_len)
loader = DataLoader(dataset, batch_size=8, shuffle=True)

## Define a Tiny JEPA-like Model

Architecture used:

- **Embedding layer**: maps each token to a dense vector.
- **Context encoder**: averages visible token embeddings, then applies two linear layers with `ReLU`.
- **Target encoder**: projects the true hidden-word embedding into the same latent space.
- **Predictor**: takes the context latent and predicts the target latent.

Why this helps:

- The embedding layer stores trainable semantic signals.
- The hidden layers learn non-linear structure in the context.
- The predictor learns which latent targets are compatible with which contexts.

This is a tiny didactic model, but the logic mirrors JEPA's central idea: **predict a missing representation, not raw output details**.

In [ ]:
class ToyJEPA(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64, latent_dim=32, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        self.context_encoder = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )

        self.target_encoder = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )

        self.predictor = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim),
        )

    def mean_pool_context(self, context_ids):
        emb = self.embedding(context_ids)
        mask = (context_ids != self.pad_idx).unsqueeze(-1)
        emb = emb * mask
        lengths = mask.sum(dim=1).clamp(min=1)
        pooled = emb.sum(dim=1) / lengths
        return pooled

    def encode_context(self, context_ids):
        pooled = self.mean_pool_context(context_ids)
        return self.context_encoder(pooled)

    def encode_target(self, target_ids):
        target_emb = self.embedding(target_ids)
        return self.target_encoder(target_emb)

    def forward(self, context_ids, target_ids):
        context_latent = self.encode_context(context_ids)
        predicted_target = self.predictor(context_latent)
        true_target = self.encode_target(target_ids)
        return predicted_target, true_target

model = ToyJEPA(vocab_size=vocab_size, pad_idx=stoi[PAD]).to(device)
model

## Train in Latent Space

We minimize mean squared error between:

- the **predicted latent target**
- the **true latent target**

That is the JEPA-style part of the notebook.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
epochs = 250

for epoch in range(epochs):
    model.train()
    total_loss = 0.0

    for batch in loader:
        context_ids = batch["context_ids"].to(device)
        target_ids = batch["target_id"].to(device)

        predicted_target, true_target = model(context_ids, target_ids)
        loss = F.mse_loss(predicted_target, true_target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if (epoch + 1) % 50 == 0:
        avg_loss = total_loss / len(loader)
        print(f"epoch={epoch + 1:03d} loss={avg_loss:.4f}")

## Recover a Word by Nearest Latent Match

JEPA itself predicts a latent vector. To make the demo easier to interpret, we compare the predicted latent with the latent representation of each vocabulary word and choose the closest one by cosine similarity.

In [ ]:
@torch.no_grad()
def build_vocab_latents(model, stoi, itos, device):
    model.eval()
    ids = []
    words = []
    for idx, word in itos.items():
        if word == PAD:
            continue
        ids.append(idx)
        words.append(word)
    ids = torch.tensor(ids, dtype=torch.long, device=device)
    latents = model.encode_target(ids)
    latents = F.normalize(latents, dim=-1)
    return words, latents

@torch.no_grad()
def predict_missing_word(model, context_tokens, stoi, words, vocab_latents, device):
    model.eval()
    context_ids = [stoi[t] for t in context_tokens]
    context_ids = context_ids + [stoi[PAD]] * (max_context_len - len(context_ids))
    context_tensor = torch.tensor([context_ids], dtype=torch.long, device=device)

    context_latent = model.encode_context(context_tensor)
    predicted_target = model.predictor(context_latent)
    predicted_target = F.normalize(predicted_target, dim=-1)

    scores = predicted_target @ vocab_latents.T
    topk = torch.topk(scores[0], k=min(5, len(words)))
    return [(words[i], float(topk.values[j])) for j, i in enumerate(topk.indices.tolist())]

words, vocab_latents = build_vocab_latents(model, stoi, itos, device)

In [ ]:
test_context = "the cat climbed the".split()
predict_missing_word(model, test_context, stoi, words, vocab_latents, device)

In [ ]:
more_tests = [
    "the dog chased the".split(),
    "the child opened the".split(),
    "the fish swam in the".split(),
    "the squirrel ran up the".split(),
]

for ctx in more_tests:
    print("context:", " ".join(ctx), "___")
    print(predict_missing_word(model, ctx, stoi, words, vocab_latents, device))
    print()

## What Hidden Layers Were Used Here?

This toy model uses simple **fully connected hidden layers**:

- `Linear(embed_dim, hidden_dim)`
- `ReLU()`
- `Linear(hidden_dim, latent_dim)`

These appear in three places:

- the **context encoder**
- the **target encoder**
- the **predictor**

Why these layers help:

- The first linear layer learns a richer combination of features from the input embedding.
- `ReLU` introduces non-linearity so the model can learn more than simple averaging.
- The final linear layer maps the hidden features into the latent space where prediction happens.

In large real JEPA systems, those simple feed-forward layers are replaced by much stronger encoders, often **Transformers** for images, video, or multimodal data.

But the educational pattern is the same:

**visible context -> latent representation -> predicted latent target**

## Final Notes

This notebook is intentionally small. It teaches the main intuition behind JEPA:

- learn from context
- predict a missing representation
- recover an interpretable answer only after latent prediction

That is not the full research story, but it is a useful bridge between the idea of JEPA and code you can run in Colab or Jupyter.